In [0]:
# Requirements Installation

%pip install pyyaml openpyxl
import sys
sys.path.insert(0, '/Workspace/Repos/ddda-clinical-platform-koios/dt-drugdevelopment-dna/databricks_bundle/drugdev/silver_framework/silver_engine')

In [0]:
# Imports

from pyspark.sql import SparkSession
from datetime import datetime
import importlib.util
import sys
import os
import re
import yaml
import json
import logging
logger = logging.getLogger(__name__)
spark = SparkSession.builder.getOrCreate()

In [0]:
# ── Performance tuning ────────────────────────────────────────────────────────

spark.conf.set("spark.sql.shuffle.partitions", "32")
spark.conf.set("spark.sql.adaptive.enabled",                          "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",       "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",        "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",          "true")
spark.conf.set("spark.databricks.delta.merge.optimizeInsertOnlyMerge.enabled", "true")
spark.conf.set("spark.databricks.delta.merge.repartitionBeforeWrite.enabled",  "true")
spark.conf.set("spark.databricks.delta.merge.enableLowShuffle",       "true")
# ─────────────────────────────────────────────────────────────────────────────

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()

NOTEBOOK_DIR = f"/Workspace{notebook_path.rsplit('/', 1)[0]}"
SILVER_DIR   = os.path.abspath(f"{NOTEBOOK_DIR}/../drugdev/silver_framework")


print("Notebook Directory        :", NOTEBOOK_DIR)
print("Silver Framework Directory:", SILVER_DIR)

In [0]:
# ── Standalone Environment Variables ─────────────────────────────────────────
# Set all values here — no widget prompts required.
# Update each value to match your target Databricks environment.

CATALOG          = "dev-drugdev_da-koios-catalog"         # Unity Catalog for Silver data tables
BRONZE_SCHEMA    = "dev_drugdev_bronze"
SILVER_SCHEMA    = "dev_drugdev_silver"
METADATA_CATALOG = "dev-drugdev_da-koios-catalog"         # Unity Catalog hosting metadata tables
REGISTRY_SCHEMA  = "dev_drugdev_common"
METADATA_SCHEMA  = "dev_drugdev_common"
ENVIRONMENT      = "dev"
STUDY_ID         = ""
VENDOR           = ""                                     # Optional vendor filter for entity/study modes
PII_UNMASK_GROUP = "OG_DIP_DBricks_Dev_Drugdev_Da_Koios_Domain_Developers,OG_DIP_DBricks_Dev_Drugdev_Da_Koios_Domain_Admins,sp:dev_drugdev_da-koios_svc_git"                  # Account group that can see unmasked PII
SOURCE_BUCKET    = "exelixis-clearlake-daplex-dev-us-west-2-441447966705-raw"
RUN_DATE         = datetime.now().strftime("%Y%m%d")      # or set explicitly e.g. "20260718"

# ── Silver pipeline execution mode ───────────────────────────────────────────
MODE        = "all"  # "all" | "domain" | "entity" | "study"
DOMAIN      = ""     # required for "domain", "entity", "study"
ENTITY      = ""     # required for "entity", "study"
MAX_WORKERS = 0      # 0 = auto based on cluster size, else explicit override

# ── Derived aliases (keep these as-is) ───────────────────────────────────────
mode          = MODE.strip().lower()
domain        = DOMAIN.strip() or None
entity        = ENTITY.strip() or None
study_id      = STUDY_ID.strip().upper() or None
vendor_param  = VENDOR.strip() or None
domain_param  = domain
entity_param  = entity
study_id_param = study_id
run_date      = RUN_DATE
environment   = ENVIRONMENT
source_bucket = SOURCE_BUCKET
CONFIG_PATH   = os.path.abspath(f"{NOTEBOOK_DIR}/../../config/environments/{environment}.yaml")

if mode not in {"all", "domain", "entity", "study"}:
    raise ValueError("Invalid mode. Use one of: all, domain, entity, study")

if mode in {"domain", "entity", "study"} and not domain_param:
    raise ValueError("DOMAIN is required when MODE is domain/entity/study")
if mode in {"entity", "study"} and not entity_param:
    raise ValueError("ENTITY is required when MODE is entity/study")
if mode == "study" and not study_id_param:
    raise ValueError("STUDY_ID is required when MODE is study")

# Compute entity parallelism from active cluster capacity.

def _compute_dynamic_workers(_spark) -> tuple[int, int, int, int]:
    # Shared clusters do not allow direct access to sparkContext, so use
    # Spark conf and conservative defaults instead of executor inspection.
    try:
        cluster_workers = int(_spark.conf.get("spark.databricks.clusterUsageTags.clusterWorkers", "0"))
        logger.info(f"Detected cluster_workers={cluster_workers}")
    except Exception:
        cluster_workers = 0
    try:
        executor_cores = int(_spark.conf.get("spark.executor.cores", "4"))
        logger.info(f"Detected executor_cores={executor_cores}")
    except Exception:
        executor_cores = 4
    if cluster_workers <= 0:
        cluster_workers = 2
    total_parallel_slots = max(cluster_workers * max(executor_cores, 1), 2)
    dynamic_workers = max(2, min(16, total_parallel_slots // 2))
    return dynamic_workers, cluster_workers, executor_cores, total_parallel_slots



dynamic_entity_workers, active_workers, executor_cores, total_slots = _compute_dynamic_workers(spark)
max_workers = MAX_WORKERS
entity_workers = dynamic_entity_workers if max_workers == 0 else max_workers

# ── Propagate to Spark conf (consumed by metadata_loader.py and silver_pipeline.py) ──
spark.conf.set("drugdev.METADATA_CATALOG",       METADATA_CATALOG)
spark.conf.set("drugdev.METADATA_SCHEMA",        METADATA_SCHEMA)
spark.conf.set("drugdev.REGISTRY_SCHEMA",        REGISTRY_SCHEMA)
spark.conf.set("drugdev.CATALOG",                CATALOG)
spark.conf.set("drugdev.BRONZE_SCHEMA",          BRONZE_SCHEMA)
spark.conf.set("drugdev.SILVER_SCHEMA",          SILVER_SCHEMA)
spark.conf.set("drugdev.environment",            ENVIRONMENT)
spark.conf.set("drugdev.study_id",               STUDY_ID)
spark.conf.set("drugdev.vendor",                 VENDOR)
spark.conf.set("drugdev.PII_UNMASK_GROUP",       PII_UNMASK_GROUP)
spark.conf.set("drugdev.source_bucket",          SOURCE_BUCKET)
spark.conf.set("drugdev.RUN_DATE",               run_date)
spark.conf.set("drugdev.YML_SILVER_CONFIG_PATH", f"{SILVER_DIR}/configs/drugdev_silver_config.yaml")
spark.conf.set("drugdev.CONFIG_PATH",            CONFIG_PATH)
spark.conf.set("drugdev.entity_workers",         str(entity_workers))

METADATA_CATALOG = spark.conf.get("drugdev.METADATA_CATALOG")
METADATA_SCHEMA  = spark.conf.get("drugdev.METADATA_SCHEMA")
REGISTRY_SCHEMA  = spark.conf.get("drugdev.REGISTRY_SCHEMA")
CATALOG          = spark.conf.get("drugdev.CATALOG")
SILVER_SCHEMA    = spark.conf.get("drugdev.SILVER_SCHEMA")
environment      = spark.conf.get("drugdev.environment")

print(f"Mode               : {mode}")
print(f"Scope              : domain={domain_param or 'ALL'}, entity={entity_param or 'ALL'}, study_id={study_id_param or 'ALL'}, vendor={vendor_param or 'ALL'}")
print(f"Cluster workers    : {active_workers} executor(s)")
print(f"Executor cores     : {executor_cores}")
print(f"Parallel slots     : {total_slots}")
print(f"Entity workers     : {entity_workers} ({'auto' if max_workers == 0 else 'manual override'})")
print(f"RUN_DATE           : {run_date}")
print(f"CATALOG            : {CATALOG}")
print(f"SILVER_SCHEMA      : {SILVER_SCHEMA}")
print(f"Environment        : {environment}")
print(f"YAML path          : {spark.conf.get('drugdev.YML_SILVER_CONFIG_PATH')}")

In [ ]:
# Split Silver YAML paths (preferred mode) + legacy fallback path
spark.conf.set("drugdev.YML_SILVER_VENDOR_CONFIG_PATH", f"{SILVER_DIR}/configs/drugdev_silver_vendor_catalog.yaml")
spark.conf.set("drugdev.YML_SILVER_ENTITY_CONFIG_PATH", f"{SILVER_DIR}/configs/drugdev_silver_entity_level.yaml")

print(f"Vendor YAML path   : {spark.conf.get('drugdev.YML_SILVER_VENDOR_CONFIG_PATH')}")
print(f"Entity YAML path   : {spark.conf.get('drugdev.YML_SILVER_ENTITY_CONFIG_PATH')}")
print(f"Legacy YAML path   : {spark.conf.get('drugdev.YML_SILVER_CONFIG_PATH')}")

In [0]:
# Helper Function — dynamic module loader


def import_from_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    mod.dbutils = dbutils
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

def get_all_domains(spark) -> list:
    return [r.domain for r in spark.sql(
        f"SELECT DISTINCT domain FROM `{CATALOG}`.{REGISTRY_SCHEMA}.drugdev_silver_registry "
        f"WHERE is_active = TRUE ORDER BY domain"
    ).collect()]



In [0]:
# Performance Tuning (Delta / AQE optimisations)

spark.conf.set("spark.sql.shuffle.partitions",                                 "32")
spark.conf.set("spark.sql.adaptive.enabled",                                   "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",                "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",                 "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",                   "true")
spark.conf.set("spark.databricks.delta.merge.optimizeInsertOnlyMerge.enabled", "true")
spark.conf.set("spark.databricks.delta.merge.repartitionBeforeWrite.enabled",  "true")
spark.conf.set("spark.databricks.delta.merge.enableLowShuffle",                "true")

print("Performance tuning applied.")

In [0]:
# Create Silver Metadata Table (idempotent -- runs once per environment)
#
# drugdev_silver_registry -- one row per (study, vendor, entity)
# Silver-only columns (17). Ingestion columns live in ingestion metadata tables.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.drugdev_silver_registry (
    registry_id          STRING      COMMENT 'UUID primary key',
    domain               STRING      COMMENT 'Lowercase domain: irt, edc, ctms',
    entity               STRING      COMMENT 'Entity name: study_site, subject_summary_report',
    vendor               STRING      COMMENT 'Vendor slug: pra, medidata, 4g',
    study_id             STRING      COMMENT 'Study identifier: XL092-001',
    source_dataset_name  STRING      COMMENT 'Bronze table name matching ingestion dataset_name',
    silver_table_name    STRING      COMMENT 'Resolved Silver FQN',
    table_sensitivity    STRING      COMMENT 'sensitivity level: public, internal, confidential, restricted',
    pii_columns          STRING      COMMENT 'JSON array of PII column names',
    scd_type             STRING,
    scd_business_keys    STRING      COMMENT 'JSON array',
    canonical_cols       STRING      COMMENT 'JSON array',
    zorder_cols          STRING      COMMENT 'JSON array',
    mapping_rules        STRING      COMMENT 'JSON: canonical_col -> vendor_col (Flow 1 direction)',
    transformation_logic STRING,
    cross_entity_sql     STRING,
    validation_rules     STRING      COMMENT 'JSON validation rule configuration',
    is_active            BOOLEAN,
    environment          STRING,
    created_at           TIMESTAMP
) USING DELTA
COMMENT 'Silver metadata registry. One row per (study, vendor, entity).
         source_dataset_name matches ingestion dataset_name for cross-layer joins.
         Ingestion columns (criticality, lifecycle_status, etc.) are in ingestion metadata tables.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.pipeline_execution_metrics (
    run_id                STRING    COMMENT 'Unique execution identifier',
    job_name              STRING    COMMENT 'Databricks job or pipeline name',
    task_name             STRING    COMMENT 'Task or notebook name',
    domain                STRING    COMMENT 'Business domain (e.g. ctms, edc, irt)',
    vendor                STRING    COMMENT 'Source vendor name',
    study_id              STRING    COMMENT 'Clinical study identifier',
    layer                 STRING    COMMENT 'Pipeline layer (bronze, silver, gold)',
    status                STRING    COMMENT 'Execution status: RUNNING, SUCCESS, FAILED',
    error_message         STRING    COMMENT 'Detailed error message if execution fails',
    error_type            STRING    COMMENT 'Exception or error category',
    start_time            TIMESTAMP COMMENT 'Execution start timestamp',
    end_time              TIMESTAMP COMMENT 'Execution end timestamp',
    duration_seconds      INT       COMMENT 'Execution duration in seconds',
    execution_timestamp   TIMESTAMP COMMENT 'Timestamp when log record was created',
    execution_date        DATE      COMMENT 'Execution date',
    records_read          BIGINT    COMMENT 'Number of records read',
    records_written       BIGINT    COMMENT 'Number of records written',
    files_processed       INT       COMMENT 'Number of files processed',
    bytes_processed       BIGINT    COMMENT 'Total bytes processed',
    checkpoint_path       STRING    COMMENT 'Checkpoint path used during execution',
    triggered_by          STRING    COMMENT 'User, scheduler, workflow, or service principal'
) USING DELTA
COMMENT 'Pipeline execution audit log table capturing runtime metrics, processing statistics, execution status, errors, and operational metadata across Bronze, Silver, and Gold layers.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.alert_history (
    alert_id            STRING    COMMENT 'Unique alert identifier',
    alert_type          STRING    COMMENT 'Alert category (e.g. PIPELINE_FAILURE, DQ_FAILURE, SLA_BREACH)',
    severity            STRING    COMMENT 'Alert severity: LOW, MEDIUM, HIGH, CRITICAL',
    title               STRING    COMMENT 'Alert title',
    message             STRING    COMMENT 'Detailed alert message',
    domain              STRING    COMMENT 'Business domain (e.g. ctms, edc, irt)',
    vendor              STRING    COMMENT 'Source vendor name',
    study_id            STRING    COMMENT 'Clinical study identifier',
    entity              STRING    COMMENT 'Affected entity or dataset',
    status              STRING    COMMENT 'Alert status: OPEN, ACKNOWLEDGED, RESOLVED',
    alert_timestamp     TIMESTAMP COMMENT 'Timestamp when alert was generated',
    alert_date          DATE      COMMENT 'Alert generation date',
    resolved_at         TIMESTAMP COMMENT 'Timestamp when alert was resolved',
    resolved_by         STRING    COMMENT 'User or system that resolved the alert',
    resolution_note     STRING    COMMENT 'Resolution details and actions taken',
    teams_sent          BOOLEAN   COMMENT 'Indicates whether Microsoft Teams notification was sent',
    email_sent          BOOLEAN   COMMENT 'Indicates whether email notification was sent',
    created_at          TIMESTAMP COMMENT 'Record creation timestamp'
) USING DELTA
COMMENT 'Pipeline alert and notification log table capturing operational, data quality, SLA, and processing alerts along with severity, resolution status, notification delivery details, and audit information.'
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.dq_validation_results_log (
    validation_id          STRING    COMMENT 'Unique validation execution identifier',
    validation_timestamp   TIMESTAMP COMMENT 'Timestamp when validation was executed',
    validation_date        DATE      COMMENT 'Validation execution date',
    schema_id              STRING    COMMENT 'Associated schema registry identifier',
    domain                 STRING    COMMENT 'Business domain (e.g. ctms, edc, irt)',
    vendor                 STRING    COMMENT 'Source vendor name',
    study_id               STRING    COMMENT 'Clinical study identifier',
    layer                  STRING    COMMENT 'Pipeline layer (bronze, silver, gold)',
    schema_name            STRING    COMMENT 'Schema name being validated',
    table_name             STRING    COMMENT 'Table name being validated',
    rule_id                STRING    COMMENT 'Unique data quality rule identifier',
    rule_name              STRING    COMMENT 'Human-readable validation rule name',
    rule_type              STRING    COMMENT 'Validation rule type (e.g. null_check, uniqueness, referential_integrity)',
    column_name            STRING    COMMENT 'Column being validated',
    severity               STRING    COMMENT 'Validation severity: LOW, MEDIUM, HIGH, CRITICAL',
    mode                   STRING    COMMENT 'Validation mode: ERROR, WARN, AUDIT',
    result                 STRING    COMMENT 'Validation result: PASS, FAIL, WARNING',
    record_count           BIGINT    COMMENT 'Total records evaluated',
    failed_record_count    BIGINT    COMMENT 'Number of records that failed validation',
    pass_rate_pct          DOUBLE    COMMENT 'Percentage of records passing validation',
    sample_failed_records  STRING    COMMENT 'JSON string containing sample failed records'
) USING DELTA
COMMENT 'Data quality validation results table capturing rule execution outcomes, validation metrics, pass/fail status, failure samples, and audit information across Bronze, Silver, and Gold layers.'
""")

print("Silver metadata table ready.")
print(f"  `{METADATA_CATALOG}`.{METADATA_SCHEMA}.drugdev_silver_registry")
print(f"  `{METADATA_CATALOG}`.{METADATA_SCHEMA}.pipeline_execution_metrics")
print(f"  `{METADATA_CATALOG}`.{METADATA_SCHEMA}.alert_history")
print(f"  `{METADATA_CATALOG}`.{METADATA_SCHEMA}.dq_validation_results_log")

In [0]:
# Load Silver Metadata from YAML → metadata tables

silver_metadata_loader = import_from_path(
    "silver_metadata_loader",
    f"{SILVER_DIR}/metadata_service/metadata_loader.py"
)

silver_metadata_loader.main()

print("Silver metadata loaded successfully.")

In [0]:
# Verification — inspect loaded metadata

print("\n── drugdev_silver_registry ───────────────────────────────────────────")
display(spark.sql(f"""
    SELECT
        registry_id,
        domain,
        entity,
        vendor,
        study_id,
        source_dataset_name,
        silver_table_name,
        scd_type,
        scd_business_keys,
        canonical_cols,
        mapping_rules,
        is_active,
        environment,
        created_at
    FROM `{METADATA_CATALOG}`.{METADATA_SCHEMA}.drugdev_silver_registry
    WHERE environment = '{environment}'
    ORDER BY domain, entity, study_id
"""))

In [ ]:
# Dependency check — required metadata lookup tables must exist before Silver execution

required_metadata_tables = [
    "country_region_mapping",
    "planisware_exs_ref",
    "site_parent_mapping",
]

metadata_namespace = f"`{METADATA_CATALOG}`.{METADATA_SCHEMA}"
missing_metadata_tables = []

for table_name in required_metadata_tables:
    table_exists = spark.sql(
        f"SHOW TABLES IN {metadata_namespace} LIKE '{table_name}'"
    ).count() > 0
    if not table_exists:
        missing_metadata_tables.append(table_name)

if missing_metadata_tables:
    missing_csv = ", ".join(missing_metadata_tables)
    raise RuntimeError(
        "Silver dependency check failed. Missing required metadata table(s) "
        f"in {metadata_namespace}: {missing_csv}"
    )

print("Silver dependency check passed. Required metadata tables are present:")
for table_name in required_metadata_tables:
    print(f"  - {metadata_namespace}.{table_name}")

In [0]:
# Run Silver Pipeline — Bronze → Silver transformation + SCD2 merge
#
# Adapted from Flow 1: processes all (domain, entity) pairs in drugdev_silver_registry
# in parallel using ThreadPoolExecutor. Each entity unions all study/vendor rows
# into a staging table then merges into the Silver Delta table via SCD2.



if mode == "all":
    domains = get_all_domains(spark)
    run_plan = [(d, None, None, None) for d in domains]
    print(f"=== Silver Pipeline: {len(domains)} active domain(s): {domains} ===")
elif mode == "domain":
    if not domain_param:
        raise ValueError("'domain' is required when mode='domain'")
    run_plan = [(domain_param, None, None, None)]
    print(f"=== Silver Pipeline: single domain run: {domain_param} ===")
elif mode == "entity":
    if not domain_param or not entity_param:
        raise ValueError("'domain' and 'entity' are required when mode='entity'")
    run_plan = [(domain_param, entity_param,None, vendor_param)]
    print(
        "=== Silver Pipeline: single entity run: "
        f"domain={domain_param}, entity={entity_param}, vendor={vendor_param or 'ALL'} ==="
    )
else:
    if not domain_param or not entity_param or not study_id_param:
        raise ValueError(
            "'domain', 'entity', and 'study_id' are required when mode='study'"
        )
    run_plan = [(domain_param, entity_param, study_id_param, vendor_param)]
    print(
        "=== Silver Pipeline: single study run: "
        f"domain={domain_param}, entity={entity_param}, study_id={study_id_param}, "
        f"vendor={vendor_param or 'ALL'} ==="
    )


alert_notifier = import_from_path(
    "alert_notifier",
    f"{SILVER_DIR}/silver_engine/alert_notifier.py"
)

silver_pipeline = import_from_path(
    "silver_pipeline",
    f"{SILVER_DIR}/silver_engine/silver_pipeline.py"
)

cfg      = {} # later needs to be picjked up from spark config
notifier = alert_notifier.AlertNotifier(cfg)

pipeline       = silver_pipeline.SilverPipeline(spark)
all_results    = {}
failed_domains = []
failed_details = []

for domain, entity, study_id, vendor in run_plan:
    label = f"{domain}" if not entity else f"{domain}/{entity}"
    if study_id:
        label = f"{label}/{study_id}"
    print(f"\n--- Running Silver for: {label} ---")
    try:
        if entity:
            result = pipeline.run_entity(
                domain=domain,
                vendor=vendor,
                entity=entity,
                study_id=study_id,
            )
            results = {entity: result}
        else:
            results = pipeline.run_domain(domain, entity_workers_override=entity_workers)
        all_results[domain] = results
        # Detect DQ failures — entities that passed transform but failed DQ
        for entity_name, r in results.items():
            dq_rate = r.get('dq_pass_rate', 100)
            if dq_rate < 100 and r.get('status') == 'SUCCESS':
                logging.warning("DQ degraded for %s/%s: pass_rate=%.1f%%", domain, entity_name, dq_rate)
    except Exception as exc:
        logging.error("Silver failed for %s: %s", label, exc, exc_info=True)
        failed_domains.append(label)
        failed_details.append(f"{label}: {exc}")
        all_results[domain] = {'ERROR': {'status': 'FAILED', 'error': str(exc)}}

print("\n=== Silver Results ===")
for domain, results in all_results.items():
    for entity, r in results.items():
        rows    = r.get('rows', 0)
        dq_rate = r.get('dq_pass_rate', 0)
        status  = r.get('status', 'UNKNOWN')
        icon    = '✓' if status == 'SUCCESS' else '✗'
        print(f"  {icon} {domain}/{entity}: {status}  rows={rows}  dq={dq_rate:.1f}%")

if failed_domains:
    # ── Recovery hint: silver reads from bronze — last successfully ingested
    # data is already in the bronze Delta tables.  On the next scheduled run
    # the failed domains will be retried automatically from that watermark.
    # For immediate recovery: re-run this task manually after fixing the cause.
    msg = (
        f"Silver pipeline failed for {len(failed_domains)} domain(s): {failed_domains}.\n"
        f"Details:\n" + "\n".join(f"  • {d}" for d in failed_details) +
        "\n\nRecovery: Silver reads from the current Bronze state. "
        "Re-run this task after resolving the error — no bronze re-ingestion needed."
    )
    try:
        notifier.send_alert(
            alert_type='PIPELINE_FAILURE', severity='CRITICAL',
            title=f"Silver Pipeline Failure [{cfg.get('env','dev')}]",
            message=msg[:400],
            context={'failed_domains': failed_domains},
        )
    except Exception as alert_exc:
        logging.warning("Failed to send silver failure alert: %s", alert_exc)
    raise RuntimeError(msg)

In [0]:
# Run Summary — current row counts per active Silver entity

domain_filter = f"AND domain = '{domain}' AND entity = '{entity}'" if (domain and entity) else ""

active_entities = spark.sql(f"""
    SELECT DISTINCT domain, entity, silver_table_name, scd_type, scd_business_keys
    FROM `{METADATA_CATALOG}`.{METADATA_SCHEMA}.drugdev_silver_registry
    WHERE is_active = TRUE
      AND environment = '{environment}'
      {domain_filter}
    ORDER BY domain, entity
""").collect()

summary_rows = []
for row in active_entities:
    try:
        current_count = spark.sql(f"""
            SELECT COUNT(*) AS cnt
            FROM {row['silver_table_name']}
            WHERE _scd_is_current = TRUE
        """).collect()[0]["cnt"]
        summary_rows.append({
            "domain":        row["domain"],
            "entity":        row["entity"],
            "silver_table":  row["silver_table_name"],
            "scd_type":      row["scd_type"],
            "current_rows":  current_count,
        })
    except Exception as e:
        summary_rows.append({
            "domain":        row["domain"],
            "entity":        row["entity"],
            "silver_table":  row["silver_table_name"],
            "scd_type":      row["scd_type"],
            "current_rows":  f"ERROR: {e}",
        })

if summary_rows:
    summary_df = spark.createDataFrame(summary_rows)
    print("\n── Silver Run Summary ────────────────────────────────────────────────")
    display(summary_df)
else:
    print("No active Silver entities found in registry.")